# 🏥 Diabetes Risk Prediction - Early Detection System  
## Notebook 2: Feature Engineering & Data Preparation

**Author:** Cephas Adams Kumah  
**Dataset:** CDC Diabetes Health Indicators - BRFSS 2015  
**Input:** `diabetes_binary_health_indicators_BRFSS2015.csv`  
**Output:** `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`

---

| Step | Description |
|---|---|
| 1. Load Data | Reload the cleaned CDC diabetes dataset prepared in Notebook 1 |
| 2. Feature Selection | Separate the target variable from the 21 predictor features |
| 3. Train/Test Split | Apply a stratified 80/20 split to preserve the diabetes class balance |
| 4. Feature Scaling | Apply `StandardScaler` for models that are sensitive to feature scale, such as Logistic Regression |
| 5. Save Prepared Data | Export training and testing datasets for model development in Notebook 3 |

## 1. Load Data

We reload the raw CDC BRFSS 2015 Diabetes Health Indicators dataset and reapply the duplicate removal step from Notebook 1. This ensures that the feature engineering pipeline is fully reproducible from the original source file.

By starting from the raw dataset and applying the same cleaning logic, we avoid depending on manually saved intermediate files and make the notebook easier to rerun, audit, and share.

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

#Load and deduplicate
df = pd.read_csv('../data/diabetes_binary_health_indicators_BRFSS2015.csv')
df = df.drop_duplicates().reset_index(drop=True)

print("=" * 40)
print(f" Rows Loaded: {len(df):,}")
print(f" Columns: {df.shape[1]}")
print(f" Mising: {df.isnull().sum().sum()}")
print("="  * 40)

 Rows Loaded: 229,474
 Columns: 22
 Mising: 0


## 2. Feature Selection

All **21 predictor features** are retained for modelling. The CDC BRFSS 2015 Diabetes Health Indicators dataset was designed around variables with clinical, behavioural, lifestyle, and socioeconomic relevance to diabetes risk.

No features are dropped at this stage because:

- All 21 features have measurable association with the target variable.
- Some features may provide useful signal when combined with other variables, even if their individual correlation is weak.
- Tree-based models such as XGBoost can handle lower-importance features effectively and rank them during model training.
- Removing features too early could risk losing useful predictive information before model evaluation.

The target variable, `Diabetes_binary`, is separated from the feature matrix so that the model can learn patterns from the predictors without data leakage.

In [8]:
#separate features and target
X = df.drop('Diabetes_binary', axis=1)
y = df['Diabetes_binary']

print("=" * 40)
print(f"  Features (X): {X.shape}")
print(f"  Target   (y): {y.shape}")
print(f"\n  Feature list:")
for i, col in enumerate(X.columns, 1):
    print(f" {i:2} . {col}")
print("=" * 40)

  Features (X): (229474, 21)
  Target   (y): (229474,)

  Feature list:
  1 . HighBP
  2 . HighChol
  3 . CholCheck
  4 . BMI
  5 . Smoker
  6 . Stroke
  7 . HeartDiseaseorAttack
  8 . PhysActivity
  9 . Fruits
 10 . Veggies
 11 . HvyAlcoholConsump
 12 . AnyHealthcare
 13 . NoDocbcCost
 14 . GenHlth
 15 . MentHlth
 16 . PhysHlth
 17 . DiffWalk
 18 . Sex
 19 . Age
 20 . Education
 21 . Income


## 3. Train / Test Split

An **80/20 stratified split** is used to divide the dataset into training and test sets.

**Why stratified?**  
The target variable is imbalanced, with approximately **84.7% No Diabetes** and **15.3% Diabetes** cases. A normal random split could place an uneven proportion of diabetic cases into either the training or test set.

Stratification ensures that both the training and test sets preserve the original class distribution. This gives the model a representative view of both classes during training and allows for a fairer evaluation on the test set.

| Set | Size | Purpose |
|---|---:|---|
| Training Set | 80% (~183,579 rows) | Used by the model to learn patterns |
| Test Set | 20% (~45,895 rows) | Held back for unbiased model evaluation |

The split is also controlled using a fixed `random_state` to make the results reproducible.

In [12]:
#Stratified train/test split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("=" * 40)
print(f"  Training set: {X_train.shape[0]:,} rows")
print(f"  Test set:     {X_test.shape[0]:,} rows")
print(f"\n  Training class balance:")
print(f"    No Diabetes: "
      f"{(y_train==0).sum():,} "
      f"({(y_train==0).mean()*100:.1f}%)")
print(f"    Diabetes:    "
      f"{(y_train==1).sum():,} "
      f"({(y_train==1).mean()*100:.1f}%)")
print(f"\n  Test class balance:")
print(f"    No Diabetes: "
      f"{(y_test==0).sum():,} "
      f"({(y_test==0).mean()*100:.1f}%)")
print(f"    Diabetes:    "
      f"{(y_test==1).sum():,} "
      f"({(y_test==1).mean()*100:.1f}%)")
print("=" * 40)

  Training set: 183,579 rows
  Test set:     45,895 rows

  Training class balance:
    No Diabetes: 155,501 (84.7%)
    Diabetes:    28,078 (15.3%)

  Test class balance:
    No Diabetes: 38,876 (84.7%)
    Diabetes:    7,019 (15.3%)


## 4. Feature Scaling

`StandardScaler` is applied to normalise the feature values for the Logistic Regression baseline model.

**Why scale?**

- Logistic Regression is sensitive to feature magnitude. Without scaling, variables with larger ranges, such as `BMI`, may have a stronger influence than variables with smaller ranges, such as `Income`.
- XGBoost does **not** require scaling because tree-based models split data using thresholds and are generally scale-invariant.
- The scaler is fitted on the training data only, then used to transform both the training and test sets. This prevents information from the test set leaking into the scaling parameters.

This approach allows us to prepare a scaled version of the dataset for Logistic Regression while keeping the original unscaled features available for XGBoost.

In [13]:
#Scale features for logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, 
                              columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled,
                             columns=X_test.columns)
print("=" * 40)
print("  Scaling complete")
print(f"  Mean (BMI after scaling):  "
      f"{X_train_scaled['BMI'].mean():.4f}")
print(f"  Std  (BMI after scaling):  "
      f"{X_train_scaled['BMI'].std():.4f}")
print("=" * 40)
                            

  Scaling complete
  Mean (BMI after scaling):  -0.0000
  Std  (BMI after scaling):  1.0000


## 5. Save Datasets

The prepared datasets are exported for use in Notebook 3: Model Training & Evaluation.

Two versions of the feature sets are saved:

- **Unscaled datasets** for XGBoost, since tree-based models are scale-invariant.
- **Scaled datasets** for the Logistic Regression baseline model, since it is sensitive to feature magnitude.

The target files are saved separately so the same `y_train` and `y_test` labels can be reused across both model types.

| File | Purpose |
|---|---|
| `X_train.csv` | Unscaled training features for XGBoost |
| `X_test.csv` | Unscaled test features for XGBoost |
| `X_train_scaled.csv` | Scaled training features for Logistic Regression |
| `X_test_scaled.csv` | Scaled test features for Logistic Regression |
| `y_train.csv` | Training target labels |
| `y_test.csv` | Test target labels |

In [14]:
# Save unscaled for XGBoost
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

# Save scaled for Logistic Regression
X_train_scaled.to_csv('../data/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/X_test_scaled.csv', index=False)

print("=" * 40)
print("  Files saved:")
print("    X_train.csv        (XGBoost)")
print("    X_test.csv         (XGBoost)")
print("    y_train.csv")
print("    y_test.csv")
print("    X_train_scaled.csv (Logistic Regression)")
print("    X_test_scaled.csv  (Logistic Regression)")
print("=" * 40)

  Files saved:
    X_train.csv        (XGBoost)
    X_test.csv         (XGBoost)
    y_train.csv
    y_test.csv
    X_train_scaled.csv (Logistic Regression)
    X_test_scaled.csv  (Logistic Regression)


---
## ✅ Feature Engineering Complete

| Output | Rows | Columns |
|---|---:|---:|
| `X_train` | 183,579 | 21 |
| `X_test` | 45,895 | 21 |
| `y_train` | 183,579 | 1 |
| `y_test` | 45,895 | 1 |

**Class balance preserved:** Both training and test sets maintain the original **84.7% / 15.3%** class split.  
**Scaling:** Applied only for the Logistic Regression baseline model.  
**No features dropped:** All **21 predictor features** are retained for modelling.  
**Data leakage avoided:** The scaler was fitted on the training set only, then applied to both training and test features.

> **Next:** Open `03_modelling.ipynb` to train and compare Logistic Regression and XGBoost models with SHAP explainability.